In [ ]:
# Lexical and Content Analysis
#installing necessary packages
import sys

!pip install pandas numpy tldextract python-whois
!pip install bs4

import os
import sys
import re
import pandas as pd
import zipfile
import numpy as np
from os.path import splitext
import ipaddress
import tldextract
import math
import statistics
import whois
import datetime
import urllib
from urllib.parse import urlparse, parse_qs

In [7]:
path1 = "phishing.csv"
df1 = pd.read_csv(path1)
df1["label"] = 0
df1.head() # label 0 indicates a phishing url

,_id,assets_downloaded,brands,domain,features.css,features.html,features.text,folder_path,language,protocol,...,security_valid_to,url,whois_domain_age,whois_raw_text,whois_registrar,whois_registrar_url,whois_registry_created_at,whois_registry_expired_at,whois_registry_updated_at,label
0,64038b0e2e9df665fc7353ff,0.700000,"[""microsoft""]",compactdrivesolu.blob.core.windows.net,"{""box-sizing"": [""inherit"", ""border-box""], ""mar...","[""html"", ""head"", ""meta"", ""title"", ""style"", ""li...",\n Sharing Link Validation\n \n Verify Your Id...,phishing/64038b0e2e9df665fc7353ff,en,http/1.1,...,2023-12-23T11:53:59.000Z,https://compactdrivesolu.blob.core.windows.net...,10067.0,Domain Name: WINDOWS.NET\r\n Registry Dom...,NaN,http://www.markmonitor.com,1995-08-10T04:00:00.000Z,2023-06-04T16:06:16.000Z,2022-05-09T19:17:07.000Z,0
1,640393d7a8281bcf8be471a4,0.857143,"[""microsoft""]",bafkreifefpvr26zfog4s27pxbtd3tqmtatd34e374cvv4...,"{""margin"": [""0""], ""font-family"": [""Raavi"", ""Mi...","[""html"", ""head"", ""script"", ""meta"", ""title"", ""m...",\n Sign in to your account\n \n Sign in\n \n E...,phishing/640393d7a8281bcf8be471a4,en,h2,...,2023-04-10T17:16:23.000Z,https://bafkreifefpvr26zfog4s27pxbtd3tqmtatd34...,2199.0,Domain Name: dweb.link\r\nRegistry Domain ID: ...,NaN,www.cscglobal.com,2017-02-24T01:05:26.675Z,2024-02-24T01:05:26.675Z,2023-01-23T21:09:56.296Z,0
2,640394d3a8281bcf8be471a8,0.500000,"[""ups""]",ups-trackid728912.is-certified.com,"{""box-sizing"": [""border-box""], ""text-rendering...","[""html"", ""head"", ""meta"", ""title"", ""meta"", ""met...",\n Global Shipping & Logistics Services | UPS ...,phishing/640394d3a8281bcf8be471a8,en,http/1.1,...,NaN,http://ups-trackid728912.is-certified.com/Find...,5901.0,Domain Name: IS-CERTIFIED.COM\r\n Registr...,NaN,http://www.tucows.com,2007-01-04T14:10:49.000Z,2024-01-04T14:10:49.000Z,2022-12-06T06:48:20.000Z,0
3,64039cc2a8281bcf8be471b5,1.000000,"[""microsoft""]",southcoastaletrail.net.au,"{""height"": [""44px"", ""100%"", ""24px"", ""40px"", ""3...","[""html"", ""head"", ""meta"", ""title"", ""link"", ""bod...",\n Sharing Link Validation\n \n Onedrive\n \n ...,phishing/64039cc2a8281bcf8be471b5,en,h2,...,2023-10-30T23:59:59.000Z,https://southcoastaletrail.net.au/original/tec...,NaN,WHOIS LIMIT EXCEEDED\n,NaN,NaN,NaN,NaN,NaN,0
4,6403a15fa8281bcf8be471c4,1.000000,"[""dhl""]",sksadesign.com,"{""width"": [""55px"", ""7px"", ""240px"", ""360px"", ""4...","[""html"", ""head"", ""meta"", ""meta"", ""title"", ""lin...",\n DHL\n \n 專業及可靠的付運服務\n \n 客户服务\n \n 全天候24小時客...,phishing/6403a15fa8281bcf8be471c4,ko,http/1.1,...,2023-04-16T04:36:29.000Z,https://sksadesign.com/@/GlobalSources/,3665.0,Domain Name: SKSADESIGN.COM\r\n Registry ...,NaN,http://www.publicdomainregistry.com,2013-01-17T06:47:49.000Z,2024-01-17T06:47:49.000Z,2022-12-19T06:11:52.000Z,0


In [8]:
path2 = "not-phishing.csv"
df2 = pd.read_csv(path2)
df2["label"] = 1
df2.head() # label 1 indicates a non phishing url

,_id,assets_downloaded,domain,features.css,features.html,features.text,folder_path,language,protocol,remote_ip_address,...,security_valid_to,url,whois_domain_age,whois_raw_text,whois_registrar,whois_registrar_url,whois_registry_created_at,whois_registry_expired_at,whois_registry_updated_at,label
0,642ea80461bb656e3eb3ee0d,0.890756,edition.cnn.com,"{""color"": [""#555555"", ""#68B631"", ""#262626"", ""#...","[""html"", ""head"", ""meta"", ""meta"", ""meta"", ""meta...","\n CNN International - Breaking News, US News,...",phishing/642ea80461bb656e3eb3ee0d,en,h2,2a04:4e42:200::773,...,2024-01-10T19:19:19.000Z,https://edition.cnn.com/,10778.0,Domain Name: CNN.COM\r\n Registry Domain ...,NaN,http://cscdbs.com,1993-09-22T00:00:00.000Z,2026-09-21T00:00:00.000Z,2020-10-20T13:09:44.000Z,1
1,642ea9b761bb656e3eb3ee0f,0.956989,booking.miramonti.aurturist.com,"{""-webkit-box-sizing"": [""border-box""], ""box-si...","[""html"", ""head"", ""style"", ""style"", ""style"", ""s...",\n Aurturist Miramonti S. Candido\n \n Ihr Jav...,phishing/642ea9b761bb656e3eb3ee0f,de,h2,135.125.240.140,...,2023-06-25T09:01:01.000Z,https://booking.miramonti.aurturist.com/,2594.0,Domain Name: AURTURIST.COM\r\n Registry D...,NaN,http://www.ionos.com,2016-02-18T09:34:35.000Z,2024-02-18T09:34:35.000Z,2020-02-10T07:14:18.000Z,1
2,642eaa1961bb656e3eb3ee11,0.916667,www.cours2soutien.fr,"{""background"": [""#EFF1F2"", ""transparent""], ""bo...","[""html"", ""head"", ""meta"", ""meta"", ""meta"", ""meta...",\n Besoin de cours de soutien scolaire à Saint...,phishing/642eaa1961bb656e3eb3ee11,fr,h2,34.117.168.233,...,2023-06-25T09:01:12.000Z,https://www.cours2soutien.fr/,390.0,%%\n%% This is the AFNIC Whois server.\n%%\n%%...,NaN,NaN,2022-03-02T08:36:55.000Z,2024-03-02T08:36:55.000Z,2023-01-27T09:52:07.813Z,1
3,642eaa7c61bb656e3eb3ee14,0.924242,canva-note.com,"{""box-sizing"": [""border-box"", ""content-box""], ...","[""html"", ""head"", ""meta"", ""meta"", ""meta"", ""meta...",\n トップページ\n \n TOP\n \n Canva使い方\n \n オンラインレッス...,phishing/642eaa7c61bb656e3eb3ee14,ja,h2,160.251.71.153,...,2023-05-26T06:52:03.000Z,https://canva-note.com/,168.0,Domain Name: CANVA-NOTE.COM\r\n Registry ...,NaN,http://gmo.jp,2022-10-10T02:26:20.000Z,2023-10-10T02:26:20.000Z,2022-10-10T11:26:22.000Z,1
4,642eaa9a61bb656e3eb3ee16,0.750000,ww1.learningplusonsemi.com,"{""font-family"": [""Arial"", ""sans-serif""], ""heig...","[""html"", ""head"", ""meta"", ""meta"", ""link"", ""titl...",\n Learningplusonsemi.com\n \n Learningplusons...,phishing/642eaa9a61bb656e3eb3ee16,en,http/1.1,63.141.242.46,...,NaN,http://ww1.learningplusonsemi.com/,5.0,Domain Name: LEARNINGPLUSONSEMI.COM\r\n R...,NaN,http://www.godaddy.com,2023-03-21T15:28:27.000Z,2024-03-21T15:28:27.000Z,2023-03-21T15:28:27.000Z,1


In [9]:
df1.columns

Index(['_id', 'assets_downloaded', 'brands', 'domain', 'features.css',
       'features.html', 'features.text', 'folder_path', 'language', 'protocol',
       'remote_ip_address', 'remote_ip_asn', 'remote_ip_country',
       'remote_ip_domain', 'remote_ip_isp', 'remote_ip_isp_org', 'scan_date',
       'security_issuer', 'security_protocol', 'security_state',
       'security_valid_from', 'security_valid_to', 'url', 'whois_domain_age',
       'whois_raw_text', 'whois_registrar', 'whois_registrar_url',
       'whois_registry_created_at', 'whois_registry_expired_at',
       'whois_registry_updated_at', 'label'],
      dtype='object')

In [10]:
df2.columns

Index(['_id', 'assets_downloaded', 'domain', 'features.css', 'features.html',
       'features.text', 'folder_path', 'language', 'protocol',
       'remote_ip_address', 'remote_ip_asn', 'remote_ip_country',
       'remote_ip_domain', 'remote_ip_isp', 'remote_ip_isp_org', 'scan_date',
       'security_issuer', 'security_protocol', 'security_state',
       'security_valid_from', 'security_valid_to', 'url', 'whois_domain_age',
       'whois_raw_text', 'whois_registrar', 'whois_registrar_url',
       'whois_registry_created_at', 'whois_registry_expired_at',
       'whois_registry_updated_at', 'label'],
      dtype='object')

In [14]:
drop_columns = [
    '_id', 'assets_downloaded', 'brands', 'remote_ip_address',
    'remote_ip_asn', 'remote_ip_country', 'remote_ip_domain',
    'remote_ip_isp', 'remote_ip_isp_org', 'scan_date',
    'security_issuer', 'security_protocol', 'security_state',
    'security_valid_from', 'security_valid_to', 'whois_domain_age',
    'whois_raw_text', 'whois_registrar', 'whois_registrar_url',
    'whois_registry_created_at', 'whois_registry_expired_at',
    'whois_registry_updated_at'
]

df1_lexical_content = df1.drop(columns=drop_columns)

In [15]:
drop_columnss = [
    '_id', 'assets_downloaded', 'language', 'remote_ip_address',
    'remote_ip_asn', 'remote_ip_country', 'remote_ip_domain',
    'remote_ip_isp', 'remote_ip_isp_org', 'scan_date',
    'security_issuer', 'security_protocol', 'security_state',
    'security_valid_from', 'security_valid_to', 'whois_domain_age',
    'whois_raw_text', 'whois_registrar', 'whois_registrar_url',
    'whois_registry_created_at', 'whois_registry_expired_at',
    'whois_registry_updated_at'
]

df2_lexical_content = df2.drop(columns=drop_columnss)

In [16]:
df1_lexical_content.head()

,domain,features.css,features.html,features.text,folder_path,language,protocol,url,label
0,compactdrivesolu.blob.core.windows.net,"{""box-sizing"": [""inherit"", ""border-box""], ""mar...","[""html"", ""head"", ""meta"", ""title"", ""style"", ""li...",\n Sharing Link Validation\n \n Verify Your Id...,phishing/64038b0e2e9df665fc7353ff,en,http/1.1,https://compactdrivesolu.blob.core.windows.net...,0
1,bafkreifefpvr26zfog4s27pxbtd3tqmtatd34e374cvv4...,"{""margin"": [""0""], ""font-family"": [""Raavi"", ""Mi...","[""html"", ""head"", ""script"", ""meta"", ""title"", ""m...",\n Sign in to your account\n \n Sign in\n \n E...,phishing/640393d7a8281bcf8be471a4,en,h2,https://bafkreifefpvr26zfog4s27pxbtd3tqmtatd34...,0
2,ups-trackid728912.is-certified.com,"{""box-sizing"": [""border-box""], ""text-rendering...","[""html"", ""head"", ""meta"", ""title"", ""meta"", ""met...",\n Global Shipping & Logistics Services | UPS ...,phishing/640394d3a8281bcf8be471a8,en,http/1.1,http://ups-trackid728912.is-certified.com/Find...,0
3,southcoastaletrail.net.au,"{""height"": [""44px"", ""100%"", ""24px"", ""40px"", ""3...","[""html"", ""head"", ""meta"", ""title"", ""link"", ""bod...",\n Sharing Link Validation\n \n Onedrive\n \n ...,phishing/64039cc2a8281bcf8be471b5,en,h2,https://southcoastaletrail.net.au/original/tec...,0
4,sksadesign.com,"{""width"": [""55px"", ""7px"", ""240px"", ""360px"", ""4...","[""html"", ""head"", ""meta"", ""meta"", ""title"", ""lin...",\n DHL\n \n 專業及可靠的付運服務\n \n 客户服务\n \n 全天候24小時客...,phishing/6403a15fa8281bcf8be471c4,ko,http/1.1,https://sksadesign.com/@/GlobalSources/,0


In [17]:
df2_lexical_content.head()

,domain,features.css,features.html,features.text,folder_path,protocol,url,label
0,edition.cnn.com,"{""color"": [""#555555"", ""#68B631"", ""#262626"", ""#...","[""html"", ""head"", ""meta"", ""meta"", ""meta"", ""meta...","\n CNN International - Breaking News, US News,...",phishing/642ea80461bb656e3eb3ee0d,h2,https://edition.cnn.com/,1
1,booking.miramonti.aurturist.com,"{""-webkit-box-sizing"": [""border-box""], ""box-si...","[""html"", ""head"", ""style"", ""style"", ""style"", ""s...",\n Aurturist Miramonti S. Candido\n \n Ihr Jav...,phishing/642ea9b761bb656e3eb3ee0f,h2,https://booking.miramonti.aurturist.com/,1
2,www.cours2soutien.fr,"{""background"": [""#EFF1F2"", ""transparent""], ""bo...","[""html"", ""head"", ""meta"", ""meta"", ""meta"", ""meta...",\n Besoin de cours de soutien scolaire à Saint...,phishing/642eaa1961bb656e3eb3ee11,h2,https://www.cours2soutien.fr/,1
3,canva-note.com,"{""box-sizing"": [""border-box"", ""content-box""], ...","[""html"", ""head"", ""meta"", ""meta"", ""meta"", ""meta...",\n トップページ\n \n TOP\n \n Canva使い方\n \n オンラインレッス...,phishing/642eaa7c61bb656e3eb3ee14,h2,https://canva-note.com/,1
4,ww1.learningplusonsemi.com,"{""font-family"": [""Arial"", ""sans-serif""], ""heig...","[""html"", ""head"", ""meta"", ""meta"", ""link"", ""titl...",\n Learningplusonsemi.com\n \n Learningplusons...,phishing/642eaa9a61bb656e3eb3ee16,http/1.1,http://ww1.learningplusonsemi.com/,1


In [18]:
df_final = pd.concat([df1_lexical_content, df2_lexical_content], ignore_index=True)
display(df_final) # final dataset that work will be done on

,domain,features.css,features.html,features.text,folder_path,language,protocol,url,label
0,compactdrivesolu.blob.core.windows.net,"{""box-sizing"": [""inherit"", ""border-box""], ""mar...","[""html"", ""head"", ""meta"", ""title"", ""style"", ""li...",\n Sharing Link Validation\n \n Verify Your Id...,phishing/64038b0e2e9df665fc7353ff,en,http/1.1,https://compactdrivesolu.blob.core.windows.net...,0
1,bafkreifefpvr26zfog4s27pxbtd3tqmtatd34e374cvv4...,"{""margin"": [""0""], ""font-family"": [""Raavi"", ""Mi...","[""html"", ""head"", ""script"", ""meta"", ""title"", ""m...",\n Sign in to your account\n \n Sign in\n \n E...,phishing/640393d7a8281bcf8be471a4,en,h2,https://bafkreifefpvr26zfog4s27pxbtd3tqmtatd34...,0
2,ups-trackid728912.is-certified.com,"{""box-sizing"": [""border-box""], ""text-rendering...","[""html"", ""head"", ""meta"", ""title"", ""meta"", ""met...",\n Global Shipping & Logistics Services | UPS ...,phishing/640394d3a8281bcf8be471a8,en,http/1.1,http://ups-trackid728912.is-certified.com/Find...,0
3,southcoastaletrail.net.au,"{""height"": [""44px"", ""100%"", ""24px"", ""40px"", ""3...","[""html"", ""head"", ""meta"", ""title"", ""link"", ""bod...",\n Sharing Link Validation\n \n Onedrive\n \n ...,phishing/64039cc2a8281bcf8be471b5,en,h2,https://southcoastaletrail.net.au/original/tec...,0
4,sksadesign.com,"{""width"": [""55px"", ""7px"", ""240px"", ""360px"", ""4...","[""html"", ""head"", ""meta"", ""meta"", ""title"", ""lin...",\n DHL\n \n 專業及可靠的付運服務\n \n 客户服务\n \n 全天候24小時客...,phishing/6403a15fa8281bcf8be471c4,ko,http/1.1,https://sksadesign.com/@/GlobalSources/,0
...,...,...,...,...,...,...,...,...,...
10390,www.the-essays.com,"{""background-image"": [""url(check-green-2120.sv...","[""html"", ""head"", ""title"", ""meta"", ""meta"", ""met...",\n Get the Essays from the Best Professional C...,phishing/6458bbe98b59edab63fd5879,NaN,h2,https://www.the-essays.com/,1
10391,academyhurricanes.co.za,"{""position"": [""absolute"", ""relative""], ""top"": ...","[""html"", ""head"", ""meta"", ""meta"", ""meta"", ""meta...",\n Hurricanes Academy\n \n Menu\n \n About Us\...,phishing/6458bd068b59edab63fd587f,NaN,http/1.1,https://academyhurricanes.co.za/,1
10392,www.financial.engineer,"{""background"": [""transparent""], ""border"": [""so...","[""html"", ""head"", ""meta"", ""meta"", ""meta"", ""meta...",\n Financial‣Engineering | Learn Wall Street M...,phishing/6458be728b59edab63fd5881,NaN,h2,https://www.financial.engineer/learn-wall-stre...,1
10393,www.etsy.com,"{""display"": [""inline-block"", ""block"", ""table-c...","[""html"", ""head"", ""meta"", ""meta"", ""meta"", ""meta...",\n Ruekrew | Etsy\n \n Einloggen\n \n GERMANY\...,phishing/6458beeb8b59edab63fd5883,NaN,h2,https://www.etsy.com/shop/Ruekrew,1


In [19]:
df_final.shape

(10395, 9)

In [20]:
# Lexical Content Analysis

import ipaddress as ip
from urllib.parse import urlparse


def GetUrlHostname(url):  # hostname extraction
    return urlparse(url).hostname

def count_dots(url: str):  # 1. Number of Dots
    return url.count('.')

def count_subdomainlevels(url: str):  # 2. Subdomain Level
    subdomain = tldextract.extract(url).subdomain
    if not subdomain:
        return 0
    else:
        return len(subdomain.split('.'))

def count_pathlevel(url: str):  # 3. Path Level
    path = urlparse(url).path
    if not path:
        return 0
    return path.count('/')

def count_urllength(url: str):  # 4. URL length
    return len(url)

def count_dash(url: str):  # 5. Number of dash symbol
    return url.count('-')

def count_atsymbol(url: str):  # 6. Number of AtSymbol
    return url.count('@')

def count_tilde(url: str):  # 7. Number of Tilde Symbol
    return url.count('~')

def count_underscore(url: str):  # 8. Number of Underscores
    return url.count('_')

def count_dollar(url: str):  # 9. Number of dollar symbol
    return url.count('$')

def count_colon(url: str):  # 10. Number of colon symbol
    return url.count(':')

def count_percent(url: str):  # 11. Number of Percent symbols
    return url.count('%')

def count_queries(url: str):  # 12. Query count
    query = urlparse(url).query
    if not query:
        return 0
    else:
        return len(query.split('&'))

def count_numeric(url: str):  # 13. Numerical character count
    count = 0
    for c in url:
        if c.isdigit():
            count = count + 1
    return count

def is_nohttps(url: str):  # 14. presence of HTTPS
    parsed = urlparse(url)
    if parsed.scheme == "":
        return 1
    return 0 if parsed.scheme.lower() == "https" else 1

def is_ip(url: str):  # 15. IP in hostname
    hostname = urlparse(url).hostname
    if hostname is None:
        return 0
    try:
        ip.ip_address(hostname)
        return 1
    except ValueError:
        return 0

def is_suffixinsubdom(url: str):  # 16. suffix(tld) in subdomain
    subdomain = tldextract.extract(url).subdomain
    if subdomain == "":
        return 0
    suf = tldextract.extract(subdomain).suffix
    if suf == '':
        return 0
    return 1

def is_https_in_hostname(url: str):  # 17. HTTPS in Hostname
    host = urlparse(url).hostname
    if not host:
        return 0
    if 'https' in host.lower():
        return 1
    return 0

def count_hostlength(url: str):  # 18. Hostname length
    hostname = GetUrlHostname(url)
    if not hostname:
        return 0
    return len(hostname)

def count_querylength(url: str):  # 19. query length
    query = urlparse(url).query
    return len(query)

def is_redirected(url: str):  # 20. Double Slash in Path
    path = urlparse(url).path
    return 1 if '//' in path else 0

def count_avg_words_length_host(url: str):  # 21. Avg wordlength in Hostname
    hostname = GetUrlHostname(url)
    if hostname == None:
        return 0
    host_words = hostname.split('.')
    return sum([len(w) for w in host_words]) / len(host_words)

def count_atsymbol_host(url: str):  # 22. Atsymbols in Hostname
    hostname = GetUrlHostname(url)
    if hostname == None:
        return 0
    return hostname.count('@')

def count_numeric_host(url: str):  # 23. Numerical Characters in Hostname
    hostname = GetUrlHostname(url)
    if hostname == None:
        return 0
    count = 0
    for c in hostname:
        if c.isdigit():
            count = count + 1
    return count

# list of sensitive words
sensitive_words = ["login", "secure", "verify", "update", "bank", "confirm",
                    "account", "webscr", "signin", "password"]

def count_sensitive_words(url: str):  # 24. Number of SensitiveWords in URL
    hostname = GetUrlHostname(url)
    return sum(1 for word in sensitive_words if word in hostname)

def url_entropy(url: str):  # 25. url entropy
    """ Shannon entropy calculation """
    prob = [float(url.count(c)) / len(url) for c in set(url)]
    return -sum(p * math.log(p, 2) for p in prob)


In [21]:
# test
url_test = df_final['url'].iloc[0]
print("Features for the first URL:")
print("count_dots:", count_dots(url_test))


##if we want to calculate for the whole url list.
##for url in df['url']:
  ##  print(f"\nFeatures for URL: {url}")
    ##print("count_dots:", count_dots(url))

Features for the first URL:
count_dots: 5


In [22]:
import requests
import re
import bs4

In [23]:
# using a fake browser User-Agent header to get a webpage

def GetResponse(url: str):
    timeout = 2  # Increased timeout to avoid too frequent timeouts
    headers = {'user-agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/107.0.0.0 Safari/537.36'}

    try:
        response = requests.get(url, headers=headers, timeout=timeout)
    except requests.exceptions.MissingSchema:
        # Try again with 'http://' prepended
        try:
            response = requests.get('http://' + url, headers=headers, timeout=timeout)
        except requests.exceptions.RequestException as e:
            print(f"Request failed: {e}")
            return None
    except requests.exceptions.RequestException as e:
        print(f"Request failed: {e}")
        return None

    return response

# This is for extracting just the domain (hostname) from a URL for further processing, logging, or validation.

def GetUrlHostname(url: str):
  if (url.startswith("http://") == False) and (url.startswith("https://") == False):
    url = "http://" + url
  parsed_url = urlparse(url)
  hostname = parsed_url.hostname  #urlparse breaks down the url into components(schema,hostname,path) and then it retrieves hostname.
  return hostname

In [24]:
#test
test_url = df_final['url'].iloc[100]
hostname = GetUrlHostname(test_url)
print(hostname)

us-server-update.i2x-tech.com


In [25]:
#test
response_test = GetResponse(test_url)
print(response_test)

<Response [404]>


In [26]:
# Content Based Features

def count_sensitive_words_url(url):     # counts the number of sensitive words mentioned in URL (lexical feature)
  cnt = 0
  sen_words = ['secure', 'account', 'verify', 'notify', 'webscr', 'login', 'ebayisapi','sign in', 'banking', 'confirm', 'click']
  for wrd in sen_words:
    cnt = cnt + url.count(wrd)
  return cnt


# instead of analyzing a URL, it scans the content of a given response (likely an HTML page or text content).
def count_sensitive_words_content(response: str):  # 26. Number of SensitiveWords in HTML
  cnt = 0
  sen_words = ['secure', 'account', 'verify', 'notify', 'webscr', 'login', 'ebayisapi','sign in', 'banking', 'confirm', 'click']
  for wrd in sen_words:
    cnt = cnt + response.count(wrd)
  return cnt


#returns the no. of hyperlinks.
#beautifulsoup just helps in traversing ie. parsing the whole html content page.
def count_hyperlinks(response: str):   # helper for hyperlink counts (not numbered)
  if response == "":
    return 0
  soup = bs4.BeautifulSoup(response, 'html.parser')
  tags = soup.find_all('a', href=True)
  return len(tags)


#calculates the percentage of external hyperlinks (links that point to different domains) in a given HTML response.
def percent_ext_hyperlinks(response: str, url: str):  # 27. External Hyperlinks percent
  if response == "":
    return 0
  ext_cnt = 0
  total = 0
  soup = bs4.BeautifulSoup(response, 'html.parser')
  tags = soup.find_all('a')

  for tag in tags:
    href = tag.get('href')
    if not href:
      continue
    href_host = urlparse(href).hostname
    url_host = urlparse(url).hostname
    # print(href, href_host)
    if href_host != None and href_host != url_host:
      ext_cnt = ext_cnt + 1
    total = total + 1
  if total == 0:
    return 0
  return (ext_cnt/total)*100


#calculates the percentage of external resource links (such as css stylesheets) in the HTML content of a webpage.
# It checks if the resources (usually linked in the <link> tag) come from external domains and returns the percentage of such external resources.
def percent_ext_resource(response: str, url: str):  # 28. External ResourceURLs percent
  if response == "":
    return 0
  ext_cnt = 0
  total = 0
  soup = bs4.BeautifulSoup(response, 'html.parser')
  tags = soup.find_all('link')
  for tag in tags:
    href = tag.get('href')
    if not href:
      continue
    href_host = urlparse(href).hostname
    url_host = urlparse(url).hostname
    if href_host != None and href_host != url_host:
      ext_cnt = ext_cnt + 1
    total = total + 1
  if total == 0:
    return 0
  return math.floor((ext_cnt/total)*100)


#check if the favicon (the small icon associated with a website) of a given webpage is hosted on a different domain than the webpage itself.
def is_ext_favicon(response : str, url : str):  # 29. Ext Favicon
  if response == "":
    return 1
  soup = bs4.BeautifulSoup(response, 'html.parser')
  tags = soup.find_all('link', attr = {'rel' : 'icon'})
  for tag in tags:
    href = tag.get('href')
    if not href:
      continue
    href_host = urlparse(href).hostname
    url_host = urlparse(url).hostname
    if url_host != None and href_host != url_host:
      return 1
  return 0


#This function helps identify forms on a webpage that are not using secure HTTPS connections
def is_insecure_forms(response: str):    # 30. Insecure Forms
  if response == "":
    return 1
  soup = bs4.BeautifulSoup(response, 'html.parser')
  tag = soup.find('form')
  if not tag:
    return 0
  to = tag.get('action')
  if not to:
    return 0
  if not to.startswith('https://'):
    return 1
  return 0


#function checks whether a webpage contains a form with a relative URL in its action attribute.
# A relative URL is a partial URL that points to a location , instead of a full URL.
def is_relative_formaction(response: str):  # 31. RelativeFormAction
  if response == "":
    return 1
  soup = bs4.BeautifulSoup(response, 'html.parser')
  tag = soup.find('form')
  if not tag:
    return 0
  to = tag.get('action')                  #action attribute determines where the form data will be submitted
  if not to:
    return 1                              #if not present then it signifies relative url therefore malicious.
  if to.startswith('/'):
    return 1
  return 0


#checks whether a given HTML response contains a form that submits to an external (non-local) URL,
#meaning the form's action attribute points to a different domain other than that url.
def is_ext_formaction(response: str, url: str):  # 32. external formaction
  if response == "":
    return 1
  soup = bs4.BeautifulSoup(response, 'html.parser')
  form = soup.find('form')
  if not form:
    return 0
  to = form.get('action')
  if not to:
    return 0
  if to[0] == '/':
    return 0
  to_host = urlparse(to).hostname
  url_host = urlparse(to).hostname
  if to_host != None and to_host != url_host:
    return 1
  return 0


#calculates the percentage of self-referencing or "null" hyperlinks in the given HTML content. The "null" hyperlinks are defined as anchor (<a>) tags whose href attribute starts with a # (indicating that they link to a section within the same page rather than to an external or different URL).
#If a URL has a significantly higher percentage of null hyperlinks compared to legitimate sites, it may increase the likelihood of being classified as malicious.
def percent_nullselfdirect_hyperlinks(response: str):  # 33. Percentage of NullSelfRedirectHyperlinks
  if response == "" :
    return 0
  cnt = 0
  total = 0
  soup = bs4.BeautifulSoup(response, 'html.parser')
  a_tags = soup.find_all('a')

  for tag in a_tags:
    total += 1
    href = tag.get('href')
    if not href:
      continue
    if href.startswith('#'):
      cnt += 1

  if total == 0:
    return 0
  return math.floor((cnt/total)*100)


#checks whether an HTML response contains any <script> tags which is often used to trigger JavaScript actions.
def is_event_mouseover(response: str):  # 34. Fake Link In StatusBar
  if response == "" :
    return 1
  if re.findall("<script>.+onmouseover.+</script>", response):
    return 1
  return 0


#check wheather the javscript content contains any prompt() function.
def is_popup_window(response: str):  # 35. popup window
  if response == "":
    return 0
  if "prompt(" in response:
    return 1
  else:
    return 0


#from response it checks rightclick is disabled or not
def is_rightclick_disabled(response: str):  # 36. Right Click Disabled
  if response == "":
    return 1
  if re.findall(r"event.button ?== ?2", response):
    return 1
  return 0


#checks wheather the html document contains missing title tag.
def is_title_missing(response: str):  # 37. Title is missing
  soup = bs4.BeautifulSoup(response, 'html.parser')
  title = soup.find('title')
  if not title:
    return 1
  if title.text == "":
    return 1
  return 0


#iframe or frame is used to embed other webpages or contents of webpages ..........often used in malicious url to embedd malicious or phishing page into a lgitimate looking site
def is_present_iframe(response: str):  # 38. Iframe Or Frame present
  soup = bs4.BeautifulSoup(response, 'html.parser')
  if soup.find('iframe') or soup.find('frame'):
    return 1
  return 0


#checks wheather domain name of url is present in the title tag
def is_domain_in_title(response: str, url: str):  # helper – domain present in title (not numbered)
  if response == "":
    return 0
  domain = tldextract.extract(url).domain
  soup = bs4.BeautifulSoup(response, 'html.parser')
  title = soup.find('title')
  if not title:
    return 0
  if domain in title:
    return 1
  return 0


def count_eval_src(text):   # 39. src_eval_cnt (count of 'eval' )
  return text.count('eval')

def count_escape_src(text):  # 40. src_escape_cnt
  return text.count('escape')

def count_exec_src(text):  # 41. src_exec_cnt
  return text.count('exec')

def count_search_src(text):  # 42. src_search_cnt
  return text.count('search')


In [27]:
# 43. Most common link domain is external
# checks whether the most frequently occurring domain in the <a> tags (hyperlinks) on a webpage matches the domain of the given URL.
# https://www.example.com -------------here domain name is example.

def is_domain_in_alink_ext(response: str, url: str):
  if response == "":
    return 1
  soup = bs4.BeautifulSoup(response, 'html.parser')
  tags = soup.find_all('a')
  doms = {}

  for tag in tags:
    href = tag.get('href')
    if not href:
      continue
    href_domain = tldextract.extract(href).domain
    if href_domain != None:
      if href_domain in doms.keys():
        doms[href_domain] += 1
      else:
        doms[href_domain] = 1

  most_freq_domain = max(doms, key=doms.get, default='')
  # print("domains in anchor tag links = ", doms)
  # print("most frequent domain =", most_freq_domain)
  host_domain = tldextract.extract(url).domain
  if host_domain == most_freq_domain:
    return 0
  return 1

In [28]:
# 44. Common Resource Domain is External
# check if the most frequent domain among all resource URLs(those used for links,images,css)is same as the url domain.

def is_domain_in_resrc_ext(response: str, url: str):
  if response == "":
    return 0
  soup = bs4.BeautifulSoup(response, 'html.parser')
  to_find = ['link', 'img', 'script']
  a = []
  for tag in to_find:
    tags = soup.find_all(tag)
    a += list(tags)
  doms = {}

  for tag in a:
    src = tag.get('src')
    if not src:
      continue
    src_domain = tldextract.extract(src).domain

    if src_domain != None:
      if src_domain in doms.keys():
        doms[src_domain] += 1
      else:
        doms[src_domain] = 1

  most_freq_domain = max(doms, key=doms.get, default='')
  host_domain = tldextract.extract(url).domain
  if host_domain == most_freq_domain:
    return 0
  return 1

In [29]:
# 45. Ratio of Common Domain in Webpage
#checks the ratio of most frequently occuring domain to the total no of urls.

def ratio_common_domain(response: str):
  if response == "":
    return 1

  soup = bs4.BeautifulSoup(response, 'html.parser')
  tags = soup.find_all('a', href=True) # only the anchor tags with href having value
                                       # recursive = [default]True

  total = 0
  refs = {}  # refs[href_value] -> frequency href_value

  for tag in tags:
    href = tag.get('href')

    if not href:
      continue

    if href in refs.keys():
      refs[href] += 1
    else:
      refs[href] = 1
    total = total + 1

  if total == 0:
    return 0

  # get the key with most frequency
  most_freq_href = max(refs, key=refs.get)

  return refs[most_freq_href] / total


In [30]:
# 46. Ratio of Common Domain in Footer
# ratio of the most frequently occurring URL (href attribute) in anchor (<a>) tags specifically within the <footer> section of a webpage, compared to the total number of anchor tags with href attributes in the <footer>.

def ratio_common_domain_footer(response: str):
  if response == "":
    return 1

  soup = bs4.BeautifulSoup(response, 'html.parser')

  footer = soup.find('footer')
  if footer == None:
    return 0

  total = 0
  refs = {}

  tags = footer.find_all('a', href=True)
  for tag in tags:
    total += 1
    href = tag.get('href')
    if not href:
      continue
    if href in refs.keys():
      refs[href] += 1
    else:
      refs[href] = 1


  if total == 0:
    return 0

  most_common_href = max(refs, key=refs.get)

  return refs[most_common_href] / total


In [31]:
# 47. Ratio of Null Links
#calculates the ratio of anchor (<a>) tags with href attributes that point to internal anchors (e.g., href="#") compared to the total number of anchor tags with href attributes

def ratio_null_links(response: str):
  if response == "":
    return 1

  soup = bs4.BeautifulSoup(response, 'html.parser')
  tags = soup.find_all('a', href=True)

  total = 0
  null_cnt = 0

  for tag in tags:
    total += 1
    href = tag.get('href')
    if not href:
      continue
    if href.startswith('#'):
      null_cnt += 1

  if total == 0:
    return 0

  return null_cnt / total

In [32]:
# 48. Ratio of Null links in Footer
def ratio_null_links_footer(response: str):
  if response == "":
    return 1

  soup = bs4.BeautifulSoup(response, 'html.parser')

  footer = soup.find('footer')
  if footer == None:
    return 0

  total = 0
  null_cnt = 0

  tags = footer.find_all('a', href=True)

  for tag in tags:
    total += 1
    href = tag.get('href')
    if not href:
      continue
    if href.startswith('#'):
      null_cnt += 1

  if total == 0:
    return 0

  return null_cnt / total

In [33]:
# 49. Anchor link present in body-tag
# checks whether there are any anchor (<a>) tags with href attributes present within the <body>

import re
import requests
import bs4

def is_alink_present_body(response: str):
    if response == "":
        return 1

    soup = bs4.BeautifulSoup(response, 'html.parser')

    body = soup.find('body')
    if body is None:
        return 0

    tags = body.find_all('a', href=True)

    if len(tags) > 0:
        return 1
    return 0


# ---------- Testing ----------
response = '''
<!DOCTYPE html>
<html>
  <head>
    <title>Page Title</title>
  </head>
  <body>
    <a href="">link</a>
    <a href="#nowhere">null link</a>
    <h1>This is a Heading</h1>
    <p>This is a paragraph.</p>
  </body>
</html>
'''

soup = bs4.BeautifulSoup(response, 'html.parser')
tags = soup.find_all('a', href=True)
print(tags)
print("Feature Output:", is_alink_present_body(response))


[<a href="">link</a>, <a href="#nowhere">null link</a>]
Feature Output: 1


In [34]:
# Third party based features

In [35]:
from datetime import datetime

In [36]:
# 50. Domain age
#THIS FUNCTION CHECKS THE AGE OF DOMAIN USING CREATION DATE AND EXPIRATION DATE ......IT RETURNS 1 IF ITS NEW ie LESS THAN 6 MONTHS OLD
#IT RETURNS 0 IF ITS OLD ie GREATER THAN 6 MONTHS OLD

def domain_age(domain_name):
  creation_date = domain_name.creation_date
  expiration_date = domain_name.expiration_date
  if (isinstance(creation_date,str) or isinstance(expiration_date,str)):
    try:
      creation_date = datetime.strptime(creation_date,'%Y-%m-%d')
      expiration_date = datetime.strptime(expiration_date,"%Y-%m-%d")
    except:
      return 1
  if ((expiration_date is None) or (creation_date is None)):
      return 1
  elif ((type(expiration_date) is list) or (type(creation_date) is list)):
      return 1
  else:
    ageofdomain = abs((expiration_date - creation_date).days)
    if ((ageofdomain/30) < 6):
      age = 1
    else:
      age = 0
  return age

In [37]:
# 51. Rank
#to determine the Google rank of a specific URL for a given search query

def get_google_rank(query, url):
    try:
        # Construct the search query URL
        search_url = f"https://www.google.com/search?q={query}&num=100"

        # Set the headers to avoid blocking by Google
        headers = {
            'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/58.0.3029.110 Safari/537.3'}

        # Sends a get request to the google search page nad then retreiving the results
        response = requests.get(search_url, headers=headers)

        # Parse the HTML content of the response using Beautiful Soup
        soup = bs4.BeautifulSoup(response.content, "html.parser")

        # Find all the search result links on the page by looking <div> elements,which is how google typically marks search results.
        search_results = soup.find_all("div", class_="g")

        # Loop through the search result links and check if the URL matches
        for i, result in enumerate(search_results):
            link = result.find("a")["href"]
            if url in link:
                # If the URL matches, return the rank
                return i+1

    except Exception as e:
        print("Error:", e)

    # If the URL is not found in the search results, return 0
    return 0

In [38]:
# This function extracts the meta description from the html content of the webpage.
#meta description is brief description of the page content often used by search engines.

In [39]:
def get_meta_content(response: str, url: str):
  if response == "" :
    print('response empty')
    return ''
  soup = bs4.BeautifulSoup(response,'html.parser')
  meta_tag = soup.find('meta', attrs={"name": "description"})   #searches for meta tag with attribute description.
  if meta_tag == None:
    return ''
  meta_content = meta_tag.get("content")                 #If meta tag is found this receives the content under content attribute
  return meta_content

In [40]:
def get_meta_keywords(response: str, url: str):
  if response == "":
    print('response empty')
    return ''

  soup = bs4.BeautifulSoup(response, 'html.parser')

  # Searches for meta tag with the attribute "keywords"
  meta_tag = soup.find('meta', attrs={"name": "keywords"})

  if meta_tag is None:
    return ''

  # Extracts the content of the meta keywords tag
  meta_keywords = meta_tag.get("content")

  return meta_keywords

In [41]:
def get_meta_authors(response: str, url: str):
  if response == "":
    print('response empty')
    return ''

  soup = bs4.BeautifulSoup(response, 'html.parser')

  # Searches for meta tag with the attribute "authors"
  meta_tag = soup.find('meta', attrs={"name": "authors"})

  if meta_tag is None:
    return ''

  # Extracts the content of the meta keywords tag
  meta_authors = meta_tag.get("content")

  return meta_authors

In [42]:
def get_meta_desc(response: str, url: str):

    meta_content = get_meta_content(response, url)
    meta_keywords = get_meta_keywords(response, url)
    meta_keywords = get_meta_keywords(response, url)

    return meta_content, meta_keywords

    #framing this function due to need but of no use

In [43]:
# Extract meta description for all URLs
df_final["meta_content"] = df_final.apply(lambda row: get_meta_content(row["features.html"], row["url"]), axis=1)

# Extract meta keywords
df_final["meta_keywords"] = df_final.apply(lambda row: get_meta_keywords(row["features.html"], row["url"]), axis=1)

# Extract meta author
df_final["meta_authors"] = df_final.apply(lambda row: get_meta_authors(row["features.html"], row["url"]), axis=1)

In [44]:
df_final[["meta_content", "meta_keywords", "meta_authors"]].head()

,meta_content,meta_keywords,meta_authors
0,,,
1,,,
2,,,
3,,,
4,,,


In [45]:
df_final["features.html"].iloc[0][:800] # dataset does not contain raw html code + no meta information

'["html", "head", "meta", "title", "style", "link", "body", "div", "div", "div", "span", "img", "div", "div", "img", "div", "div", "span", "div", "div", "div", "div", "img", "div", "span", "div", "div", "span", "div", "div", "img", "div", "input", "div", "i", "div", "center", "div", "button", "div", "span", "a", "div", "img", "script", "noscript", "i"]'

In [46]:
# this func evaluates google search rank based on meta description
def get_meta_rank(response: str, url: str):
  query = get_meta_content(response, url)
  rank = get_google_rank(query, url)
  return rank

In [47]:
#FEATURE EXTRACTOR

In [48]:
def featureExtract(url, label):
    result = []
    print(f"Extracting features for URL: {url}")

    try:
        parsed = urlparse(url)
        print(f"Type of parsed: {type(parsed)}, Value: {parsed}")
        print(f"Type of parsed.path: {type(parsed.path)}, Value: {parsed.path}")

        result.append(str(url))

        result.append(count_dots(url))  # 1
        result.append(count_subdomainlevels(url))  # 2
        result.append(count_pathlevel(url))  # 3
        result.append(count_urllength(url))  # 4
        result.append(count_dash(url))  # 5
        result.append(count_atsymbol(url))  # 6
        result.append(count_tilde(url))  # 7
        result.append(count_underscore(url))  # 8
        result.append(count_dollar(url))  # 9
        result.append(count_colon(url))  # 10
        result.append(count_percent(url))  # 11
        result.append(count_queries(url))  # 12
        result.append(count_numeric(url))  # 13
        result.append(is_nohttps(url))  # 14
        result.append(is_ip(url))  # 15
        result.append(is_suffixinsubdom(url))  # 16
        result.append(is_https_in_hostname(url))  # 17
        result.append(count_hostlength(url))  # 18
        result.append(count_querylength(url))  # 19
        result.append(is_redirected(parsed.path))  # 20
        result.append(count_avg_words_length_host(url))  # 21
        result.append(count_atsymbol_host(url))  # 22
        result.append(count_numeric_host(url))  # 23

        try:
            res = GetResponse(url).text
        except:
            res = ""

        result.append(count_sensitive_words_url(url))  # 24
        result.append(url_entropy(url)) # 25
        result.append(count_sensitive_words_content(res))  # 26
        result.append(count_hyperlinks(res)) # 27
        result.append(percent_ext_hyperlinks(res, url))  # 27
        result.append(percent_ext_resource(res, url))  # 28
        result.append(is_ext_favicon(res, url))  # 29
        result.append(is_insecure_forms(res))  # 30
        result.append(is_relative_formaction(res)) # 31
        result.append(is_ext_formaction(res, url)) # 32
        result.append(percent_nullselfdirect_hyperlinks(res))  # 33
        result.append(is_event_mouseover(res))  # 34
        result.append(is_popup_window(res)) # 35
        result.append(is_rightclick_disabled(res))  # 36
        result.append(is_title_missing(res)) # 37
        result.append(is_present_iframe(res))  # 38
        result.append(count_eval_src(res))  # 39
        result.append(count_escape_src(res))  # 40
        result.append(count_exec_src(res))  # 41
        result.append(count_search_src(res))  # 42

        result.append(is_domain_in_alink_ext(res, url))  # 43
        result.append(is_domain_in_resrc_ext(res, url))  # 44
        result.append(ratio_common_domain(res))  # 45
        result.append(ratio_common_domain_footer(res))  # 46
        result.append(ratio_null_links(res))  # 47
        result.append(ratio_null_links_footer(res))  # 48
        result.append(is_alink_present_body(res))  # 49

        domain_name = tldextract.extract(url).registered_domain
        query = urlparse(url).query

        result.append(domain_age(domain_name)) # 50
        result.append(get_google_rank(query, url)) # 51

        result.append(get_meta_rank(res, url)) # 52
        result.append(label)

        result.extend([0] * (57 - len(result)))  # Fill with zeros to match column count

    except Exception as e:
        print(f"Error occurred: {e}")

    return result


In [49]:
# Test the featureExtract function separately with an example
testing_url = df_final.url.iloc[0]
test_label = df_final.label.iloc[0]

# Check if featureExtract works as expected
features_test = featureExtract(testing_url, test_label)
print(f"Features for {testing_url}: {features_test}")

Extracting features for URL: https://compactdrivesolu.blob.core.windows.net/nedo/latoy.html
Type of parsed: <class 'urllib.parse.ParseResult'>, Value: ParseResult(scheme='https', netloc='compactdrivesolu.blob.core.windows.net', path='/nedo/latoy.html', params='', query='', fragment='')
Type of parsed.path: <class 'str'>, Value: /nedo/latoy.html
Request failed: HTTPSConnectionPool(host='compactdrivesolu.blob.core.windows.net', port=443): Max retries exceeded with url: /nedo/latoy.html (Caused by NameResolutionError("<urllib3.connection.HTTPSConnection object at 0x0000019996D62490>: Failed to resolve 'compactdrivesolu.blob.core.windows.net' ([Errno 11001] getaddrinfo failed)"))
Error occurred: 'str' object has no attribute 'creation_date'
Features for https://compactdrivesolu.blob.core.windows.net/nedo/latoy.html: ['https://compactdrivesolu.blob.core.windows.net/nedo/latoy.html', 5, 3, 2, 62, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 38, 0, 0, 6.8, 0, 0, 0, 4.247898730680087, 0, 0, 0, 0, 1,

C:\Users\Parijat Biswas\AppData\Local\Temp\ipykernel_4632\1500308368.py:70: DeprecationWarning: The 'registered_domain' property is deprecated and will be removed in the next major version. Use 'top_domain_under_public_suffix' instead, which has the same behavior but a more accurate name.
  domain_name = tldextract.extract(url).registered_domain


In [ ]:
all_features =[]

for i, row in df_final.iterrows():
    url = row["url"]
    label = row["label"]
    feats = featureExtract(url, label)

    # Ensure every result is a flat dict-like object
    if isinstance(feats, dict):
        all_features.append(feats)
    else:
        # if list/tuple, convert to dict
        feats_dict = {f"feature_{i}": v for i, v in enumerate(feats)}
        feats_dict["label"] = label
        feats_dict["url"] = url
        all_features.append(feats_dict)

# Create dataframe directly from output
features_df_final = pd.DataFrame(all_features)

# Save to CSV and Excel
features_df_final.to_csv("features_output.csv", index=False)
features_df_final.to_excel("features_output.xlsx", index=False)

print("Saved as features_output.csv and features_output.xlsx")
display(features_df_final)



In [ ]:
df = pd.read_excel("/content/features_output.xlsx")   
print("Columns found:", df.shape[1])
print("Rows:", df.shape[0])

final_column_names = [
    'url',                                   # 0
    'Number of Dots',                        # 1
    'Subdomain Level',                       # 2
    'Path Level',                            # 3
    'URL length',                            # 4
    'Number of dash symbol',                 # 5
    'Number of AtSymbol',                    # 6
    'Number of Tilde Symbol',                # 7
    'Number of Underscores',                 # 8
    'Number of dollar symbol',               # 9
    'Number of colon symbol',                # 10
    'Number of Percent symbols',             # 11
    'Query count',                           # 12
    'Numerical character count',             # 13
    'presence of HTTPS',                     # 14
    'IP in hostname',                        # 15
    'suffix(tld) in subdomain',              # 16
    'HTTPS in Hostname',                     # 17
    'Hostname length',                       # 18
    'query length',                          # 19
    'Double Slash in Path',                  # 20
    'Avg wordlength in Hostname',            # 21
    'Atsymbols in Hostname',                 # 22
    'Numerical Characters in Hostname',      # 23
    'Number of SensitiveWords in URL',       # 24
    'url entropy',                           # 25
    'Number of SensitiveWords in HTML',      # 26
    'Number of Hyperlinks',                  # 27
    'External Hyperlinks percent',           # 28
    'External ResourceURLs percent',         # 29
    'Ext Favicon',                           # 30
    'Insecure Forms',                        # 31
    'RelativeFormAction',                    # 32
    'external formaction',                   # 33
    'Percentage of NullSelfRedirectHyperlinks', # 34
    'Fake Link In StatusBar',                # 35
    'popup window',                          # 36
    'Right Click Disabled',                  # 37
    'Title is missing',                      # 38
    'Iframe Or Frame present',               # 39
    'src_eval_cnt',                          # 40
    'src_escape_cnt',                        # 41
    'src_exec_cnt',                          # 42
    'src_search_cnt',                        # 43
    'Most Common Link Domain in External',   # 44
    'Common Resource Domain in External',    # 45
    'Ratio of Common Domain in Webpage',     # 46
    'Ratio of Common Domain in Footer',      # 47
    'Ratio of Null Links',                   # 48
    'Ratio of Null Links in Footer',         # 49
    'Ratio of Null links present in body-tag', # 50
    'domain age',                            # 51
    'google rank',                           # 52
    'meta rank',                             # 53
    'label',                                 # 54
    'pad1',                                   # 55
    'pad2'                                    # 56
]

expected = len(final_column_names)
actual = df.shape[1]

if actual < expected:
    # pad missing columns with NaN
    missing = expected - actual
    for i in range(missing):
        df[f'_pad_{i+1}'] = pd.NA
    print(f"Padded with {missing} new columns to reach {expected}.")

elif actual > expected:
    # trim extra columns
    df = df.iloc[:, :expected]
    print(f"Trimmed extra columns. Now {expected} columns.")

df.columns = final_column_names
print("Column names fixed successfully!")

df.to_csv("features_final_clean.csv", index=False)
df.to_excel("features_final_clean.xlsx", index=False)

print("Saved as features_final_clean.csv and features_final_clean.xlsx")

display(df.head())